# 19 · Relative Mahalanobis distance for unfamiliar-fault detection (Stage R8)

**Question.** Does Relative Mahalanobis distance (RMD) detect unfamiliar faults better than the shared-covariance
Mahalanobis distance the decision rule uses today, with everything else unchanged?

Exploratory, development data only, **no training**: the 81 saved leave-one-class-out (LOCO) CNN-6 runs (9 withheld
classes × 3 outer folds × seeds 42, 43, 44), their saved window embeddings and saved scores. Reserved tests stay sealed.

## Background

**What the familiarity check does today.** Every recording's CNN-6 embedding is compared with the centroid of each known
class, using one covariance shared by all classes (the spread of the training windows around their own class centroid).
The smallest of these Mahalanobis distances says how familiar the recording looks; too far from every known class means
"further inspection needed".

**Why kNN lost.** An earlier LOCO analysis also tried the cosine distance to the 10th nearest training window. It was worse
on all three seeds (macro AUROC about 0.845 against 0.877 for Mahalanobis): a plain nearest-neighbour distance does not
use the shape of the classes.

**What RMD changes.** Our unfamiliar faults are *near* out-of-distribution: a withheld fault looks like the other faults,
not like noise. Part of any Mahalanobis distance is simply "how far from the bulk of all machine recordings", which every
fault shares. Relative Mahalanobis (Ren et al., 2021, "A Simple Fix to Mahalanobis Distance for Improving Near-OOD
Detection") subtracts the distance to one *background* Gaussian fitted on all known training windows together, so that
what remains is only what is specific to the nearest class: RMD = min over classes of MD_class − MD_background (squared
distances). It has no free parameter. The figure below shows the idea on synthetic 2-D points.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> draws a toy picture on synthetic 2-D points (fixed seed): two known classes, each with its own Gaussian, and the one background Gaussian fitted on all their points together (dashed). It saves the figure to <code>results/figures/r8_toy.png</code>; it uses no project data.</div>

In [ ]:
# Toy picture of the class Gaussians and the background Gaussian (synthetic points, fixed seed; no project data).
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def ellipse(ax, mean, cov, **kw):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    ax.add_patch(Ellipse(mean, 2 * 2 * np.sqrt(vals[1]), 2 * 2 * np.sqrt(vals[0]), angle=angle, fill=False, **kw))

rng = np.random.default_rng(0)
a = rng.multivariate_normal([0, 0], [[1.0, 0.6], [0.6, 0.8]], 200)
b = rng.multivariate_normal([4, 1.5], [[1.0, 0.6], [0.6, 0.8]], 200)
both = np.vstack([a, b])
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.scatter(*a.T, s=6, alpha=.5, label="known class A"); ax.scatter(*b.T, s=6, alpha=.5, label="known class B")
ellipse(ax, a.mean(0), np.cov(a.T), color="C0", lw=2); ellipse(ax, b.mean(0), np.cov(b.T), color="C1", lw=2)
ellipse(ax, both.mean(0), np.cov(both.T), color="k", lw=1.5, ls="--")
ax.scatter([2.0], [3.2], marker="*", s=180, color="C3", label="a near-OOD recording")
ax.set_xlabel("embedding direction 1 (arbitrary units)"); ax.set_ylabel("embedding direction 2 (arbitrary units)")
ax.set_title("Class Gaussians (solid) and the background Gaussian (dashed), 2σ ellipses")
ax.legend(loc="lower right", fontsize=8); ax.set_aspect("equal"); fig.tight_layout()
from ml.paths import SYSTEM_CODE_DIR
fig.savefig(SYSTEM_CODE_DIR / "results" / "figures" / "r8_toy.png", dpi=150)
plt.show()

## The declaration, fixed before any code or computation
The protocol below is quoted directly from CLAUDE.md §7 (Stage R8, committed before any code); the cell prints it so the
reader sees the method, arms, metrics, bar and stop rule were fixed first.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> reads CLAUDE.md and prints the Stage R8 protocol exactly as committed: the method, the data, the three arms, the evaluation, the acceptance bar, the outputs and the stop rule.</div>

In [ ]:
# Quote the committed Stage R8 declaration from CLAUDE.md (so the protocol shown here is the one fixed before computing).
import re
claude = (SYSTEM_CODE_DIR.parent / "CLAUDE.md").read_text()
block = claude[claude.index("**Stage R8 protocol"):claude.index("## 7-prev11.")]
print(block.strip())

## Load the runs
The run plan (seed × withheld class × outer fold) and, per run, how many recordings play each role: fit (the model's
training recordings; their windows fit the class and background Gaussians), cal (calibration: they set τ_p and the
calibrated thresholds), eval (seen-folder evaluation), foldk_known (the unseen outer fold's known recordings: the
negatives) and foldk_withheld (the withheld class in fold k: the positives).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> sets up provenance (git sha, split hash, config sha) and lists the 81 runs with, per run, the number of recordings in each role, read from the saved Stage 3C / 3C.1 recording tables (the fit recordings are counted from the saved windows).</div>

In [ ]:
# Provenance and the run plan; per-run role counts from the saved recording tables and window files.
import json
import pandas as pd
from ml.config import config_sha256, load_config
from ml.paths import DEFAULT_CONFIG, RunPaths, results_dir
from ml.rmd_eval import ARMS, ARM_NAMES, evaluate_run, pooled, reading, reproduction, run_scores, runs
from ml.loco_eval import macro
from ml.splits import FAULT_CLASSES, split_hash
from ml.utils import git_sha

cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
prov = {"git_sha": git_sha(), "split_hash": split_hash(meta), "config_sha256": config_sha256(cfg)}
print(prov)
plan = runs()
counts = []
for s, c, k in plan:
    rp = RunPaths("loco", "cnn6", s, protocol="loco", withheld=c)
    roles = pd.read_parquet(rp.scores(f"fold{k}_recording"), columns=["role"]).role.value_counts()
    fit_recs = pd.read_parquet(rp.scores(f"fold{k}_logits"), columns=["rec_id", "role"]).query("role == 'fit'").rec_id.nunique()
    counts.append({"seed": s, "withheld": c, "fold": k, "fit": fit_recs, **roles.to_dict()})
counts = pd.DataFrame(counts)
print(len(plan), "runs")
counts.drop(columns=["seed"]).groupby(["withheld", "fold"]).first().head(9)

## Reproduction check (before anything else)
Arm M recomputed from the saved window embeddings must equal the saved Stage 3C / 3C.1 recording Mahalanobis of every run
(|Δ maha| ≤ 1e-4 × maha, R7's tolerance), and arm K the saved kNN score. Any mismatch stops the stage.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> recomputes, for every one of the 81 runs, the three familiarity scores (M, K, R) from the saved window embeddings, then checks M against the saved Mahalanobis and K against the saved kNN score. The assert stops the notebook on any mismatch; the printed numbers are the largest differences found.</div>

In [ ]:
# One pass over the 81 runs: arms M, K and R from the saved embeddings; M and K must reproduce the saved 3C scores.
import time
t0 = time.perf_counter()
tables, fits, repro = {}, {}, []
for s, c, k in plan:
    rec, info = run_scores(s, c, k)
    tables[(s, c, k)], fits[(s, c, k)] = rec, info
    repro.append({"seed": s, "withheld": c, "fold": k, **reproduction(rec)})
repro = pd.DataFrame(repro)
score_seconds = time.perf_counter() - t0
print(f"scored 81 runs in {score_seconds:.0f} s; max relative |Δ maha| = {repro.max_rel_diff_maha.max():.2e}; "
      f"max |Δ knn| = {repro.max_abs_diff_knn.max():.2e}; all within tolerance: {repro.ok.all()}")
assert repro.ok.all(), repro[~repro.ok]

## The three scores on one run (sanity table)
One run (seed 42, withheld imbalance, fold 0): the scores of a few recordings of each evaluated role, and the fitted
Ledoit–Wolf shrinkages. Higher = more unfamiliar for all three arms; RMD can be negative (closer to a class than to the
background).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> shows, for one run, three recordings of each role with their saved Mahalanobis and the three recomputed scores, and the shrinkage of the class and background covariance fits.</div>

In [ ]:
# Sanity view of one run: a few recordings per role with M (= saved maha), K and R, plus the fitted shrinkages.
one = tables[(42, "imbalance", 0)]
print(fits[(42, "imbalance", 0)])
one.groupby("role").head(3)[["rec_id", "role", "class10", "pred", "msp", "maha", "M", "K", "R"]].round(4)

## Threshold-free results
Per run, AUROC and OSCR with the withheld-class recordings of fold k as positives and fold k's known recordings as
negatives (weight 1, OSCR correctness = CNN-6 top class is the true class), as in the existing LOCO analysis. The macro is
the mean over folds within a withheld class, then over the 9 classes, per seed.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> evaluates every run under every arm (threshold-free metrics and the decision rule at the matched and the calibrated thresholds; <code>ml.rmd_eval.evaluate_run</code>), then tabulates the macro AUROC and OSCR per seed and arm, with the mean over the three seeds.</div>

In [ ]:
# Evaluate every run and arm once; macro AUROC / OSCR over the 9 withheld classes, per seed, and the seed mean.
rows = []
for (s, c, k), rec in tables.items():
    for r in evaluate_run(rec):
        rows.append({"seed": s, "withheld": c, "outer_fold": k, **r})
per_run = pd.DataFrame(rows)
tf = pd.DataFrame([{"seed": s, "arm": a, "macro_auroc": macro(g, "auroc"), "macro_oscr": macro(g, "oscr")}
                   for (s, a), g in per_run.groupby(["seed", "arm"])])
tf_table = tf.pivot(index="seed", columns="arm", values=["macro_auroc", "macro_oscr"])
tf_table.loc["mean"] = tf_table.mean()
tf_table.round(4)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> shows the per-class AUROC (mean over seeds and folds) for M, K and R as grouped bars, one group per withheld class, and saves it to <code>results/figures/r8_auroc_per_class.png</code>. Higher bars mean the arm separates that withheld class from the known recordings better.</div>

In [ ]:
# Per-class AUROC (mean over the 3 folds and 3 seeds) for each arm, as grouped bars.
pc = per_run.groupby(["withheld", "arm"]).auroc.mean().unstack()[list(ARMS)]
fig, ax = plt.subplots(figsize=(10, 3.8))
pc.plot.bar(ax=ax, width=.8)
ax.set_ylabel("AUROC (withheld vs fold-k known)"); ax.set_xlabel("withheld class"); ax.set_ylim(0.4, 1.0)
ax.set_title("Per-class AUROC, mean over folds and seeds"); ax.legend([ARM_NAMES[a] for a in ARMS], fontsize=8, loc="lower left")
plt.xticks(rotation=30, ha="right"); fig.tight_layout()
fig.savefig(SYSTEM_CODE_DIR / "results" / "figures" / "r8_auroc_per_class.png", dpi=150)
plt.show()
pc.round(3)

## In the decision rule (the primary result)
The score replaces the shared Mahalanobis term in the confidence gate, everything else unchanged (τ_p, rule B's healthy
logic, the predicted class). **Primary:** τ_s at a matched known-side rejection count — the gate rejects exactly as many
fold-k known recordings as arm M does at τ_d. Pooled over the 27 runs of each seed: unknown recall on the withheld
recordings, withheld recordings called healthy (must not rise) and known recordings called the wrong fault.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> pools the matched-threshold outcomes over the 27 runs of each seed, per arm: unknown recall (share of withheld-class recordings sent to further inspection), withheld recordings called healthy, known recordings called the wrong fault, and the known-side rejections (equal across arms by construction, unless a count could not be reached).</div>

In [ ]:
# Matched known-side rejection count: pooled outcomes per seed and arm.
unreached = per_run[per_run.tau_matched.isna()]
print("runs where the matched count could not be reached:", len(unreached))
matched = pooled(per_run, "matched")
matched.round(4)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> shows the same outcomes with τ_s set the way τ_d is set today (the 0.975 order statistic on the calibration recordings): what each arm would do in practice, where the known-side rejections are no longer equal.</div>

In [ ]:
# Secondary: τ_s calibrated like τ_d (0.975 order statistic on the calibration recordings).
calibrated = pooled(per_run, "calibrated")
calibrated.round(4)

## Per-class unknown recall, M against R
Unknown recall at the matched count, per withheld class (pooled over the class's 3 folds within a seed, then the mean over
seeds). Classes where R is better or worse by 5 points or more are marked.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> computes unknown recall per withheld class for every arm and seed at the matched threshold, takes the mean over seeds, marks classes where R differs from M by 5 points or more, and draws M and R side by side (<code>results/figures/r8_recall_per_class.png</code>).</div>

In [ ]:
# Per-class unknown recall (matched count): per seed, the seed mean, and R − M with ≥ 5-point differences marked.
g = per_run.groupby(["seed", "withheld", "arm"])
cls = (g.matched_withheld_unknown.sum() / g.matched_withheld_n.sum()).unstack("arm")
cls_mean = cls.groupby(level="withheld").mean()
cls_mean["R_minus_M"] = cls_mean.R - cls_mean.M
cls_mean["mark"] = np.where(cls_mean.R_minus_M >= 0.05, "R better (≥ 5 pts)", np.where(cls_mean.R_minus_M <= -0.05, "R worse (≥ 5 pts)", ""))
fig, ax = plt.subplots(figsize=(10, 3.6))
cls_mean[["M", "R"]].plot.bar(ax=ax, width=.75, color=["C0", "C2"])
ax.set_ylabel("unknown recall (matched count)"); ax.set_xlabel("withheld class"); ax.set_ylim(0, 1)
ax.set_title("Unknown recall per withheld class, mean over seeds"); ax.legend([ARM_NAMES["M"], ARM_NAMES["R"]], fontsize=8)
plt.xticks(rotation=30, ha="right"); fig.tight_layout()
fig.savefig(SYSTEM_CODE_DIR / "results" / "figures" / "r8_recall_per_class.png", dpi=150)
plt.show()
display(cls.unstack("seed")[["M", "R"]].round(3))
cls_mean.round(3)

## Reading against the declared bar
R **helps, confirmed** only if, against M at the matched count, unknown recall is at least 5 points higher AND no more
withheld recordings are called healthy, on all three seeds; otherwise **worse** if the mean change is negative; otherwise
**does not help**. Applied mechanically:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> applies the declared bar to the matched-count table (<code>ml.rmd_eval.reading</code>) and prints the per-seed recall gain of R over M, the change in withheld recordings called healthy, and the reading.</div>

In [ ]:
# The declared reading, applied mechanically.
verdict = reading(matched)
for s, v in verdict["per_seed"].items():
    print(f"seed {s}: unknown recall R − M = {v['recall_gain'] * 100:+.1f} points; withheld called healthy R − M = "
          f"{v['healthy_change']:+d}; bar met: {v['bar_met']}")
print(f"mean recall change {verdict['mean_recall_gain'] * 100:+.1f} points  →  READING: {verdict['reading'].upper()}")

**In plain words.** Relative Mahalanobis **does not help** on these runs. At the matched known-side rejection count it changes unknown recall by +0.5, −0.4 and +0.6 points on seeds 42, 43 and 44 (mean +0.3), far below the +5-point bar, and withheld recordings called healthy fall on seeds 42 and 44 (−3, −4) but rise on seed 43 (+2). Threshold-free it is level with Mahalanobis (macro AUROC 0.875 vs 0.874 over the seeds), and kNN stays the weakest (0.846). Where R is worse: with τ_s calibrated the way τ_d is today, it sends 0.5–2.2 points fewer withheld recordings to inspection than M on every seed and calls 8 more withheld recordings healthy on seed 42. The two classes that drive the open-set gap are unchanged under every arm: withheld imbalance is almost never flagged (recall about 5 %, AUROC about 0.59) and withheld vertical misalignment stays at 16–17 % with an AUROC below 0.5, so subtracting the shared background distance does not reach them.


## Limitations
- Development data only; nothing here is an independent claim (reserved tests sealed; the final evaluation already ran).
- The thresholds come from in-sample calibration recordings of each run; the matched comparison removes threshold choice
  from the primary result but uses the same known recordings that are counted.
- The same LOCO partition, folds and saved models as Stages 3C, R4–R7; every withheld class was seen during earlier
  development of the system.
- One method is compared (RMD as published), with no tuning and no free parameter; a negative result says this
  correction does not help here, not that no relative or class-conditional distance could.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> writes the per-run table to <code>results/research/r8_per_run.csv</code> and every reported number, with provenance and the reading, to <code>results/research/r8_relative_mahalanobis.json</code>.</div>

In [ ]:
# Write the outputs: per-run CSV and the JSON summary (provenance, per seed / run / class metrics, the reading).
out = results_dir() / "research"
per_run.to_csv(out / "r8_per_run.csv", index=False)
summary = {"stage": "R8", "provenance": prov | {"score_seconds": round(score_seconds, 1)},
           "arms": ARM_NAMES, "reproduction": {"max_rel_diff_maha": float(repro.max_rel_diff_maha.max()),
                                               "max_abs_diff_knn": float(repro.max_abs_diff_knn.max()), "ok": bool(repro.ok.all())},
           "threshold_free_macro": tf.to_dict(orient="records"),
           "threshold_free_per_class": per_run.groupby(["seed", "withheld", "arm"])[["auroc", "oscr"]].mean().reset_index().to_dict(orient="records"),
           "matched": matched.to_dict(orient="records"), "calibrated": calibrated.to_dict(orient="records"),
           "matched_unreached_runs": int(len(unreached)),
           "recall_per_class_matched": cls.reset_index().to_dict(orient="records"),
           "recall_per_class_seed_mean": cls_mean.reset_index().to_dict(orient="records"),
           "fits": {f"{s}/{c}/{k}": v for (s, c, k), v in fits.items()}, "reading": verdict}
(out / "r8_relative_mahalanobis.json").write_text(json.dumps(summary, indent=1, default=float))
print("written:", (out / "r8_per_run.csv").relative_to(SYSTEM_CODE_DIR), (out / "r8_relative_mahalanobis.json").relative_to(SYSTEM_CODE_DIR))

## Outputs written
- `results/research/r8_relative_mahalanobis.json`, `results/research/r8_per_run.csv`
- Figures: `results/figures/r8_toy.png`, `r8_auroc_per_class.png`, `r8_recall_per_class.png`